In [1]:
# PySpark Imports
import pyspark
from pyspark.sql import SparkSession

# ML Classifier Imports
from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler, StringIndexer, PCA
from pyspark.ml.classification import OneVsRest
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.sql.functions import mean, col
import time
import os
import sys

In [2]:
# Initialize Spark session
spark = SparkSession.builder.appName("ce53") \
    .master("local[*]") \
    .config("spark.driver.cores", "2") \
    .config("spark.driver.memory", "14g") \
    .config("spark.executor.memory", "14g") \
    .config("spark.executor.cores", "2") \
    .config("spark.dynamicAllocation.shuffleTracking.enabled", "true") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.dynamicAllocation.minExecutors", "2") \
    .config("spark.dynamicAllocation.maxExecutors", "2") \
    .config("spark.executor.instances", "2") \
    .config("spark.kryoserializer.buffer.max", "2047m") \
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false") \
    .config("spark.executor.heartbeatInterval","11999s") \
    .config("spark.network.timeout","12000s") \
.getOrCreate()

24/04/15 02:31:43 WARN Utils: Your hostname, colin-MS-7977 resolves to a loopback address: 127.0.1.1; using 192.168.0.164 instead (on interface wlx3c52a1d3ccda)
24/04/15 02:31:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/15 02:31:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
parquet_files = ["Parquet/part-00000-1da06990-329c-4e38-913a-0f0aa39b388d-c000.snappy.parquet", "Parquet/part-00000-26e9208e-7819-451b-b23f-2e47f6d1e834-c000.snappy.parquet", 
                 "Parquet/part-00000-36240b61-b84f-4164-a873-d7973e652780-c000.snappy.parquet", "Parquet/part-00000-3f86626a-1225-47f9-a5a2-0170b737e404-c000.snappy.parquet",
                 "Parquet/part-00000-7c2e9adb-5430-4792-a42b-10ff5bbd46e8-c000.snappy.parquet", "Parquet/part-00000-b1a9fc13-8068-4a5d-91b2-871438709e81-c000.snappy.parquet",
                 "Parquet/part-00000-cbf26680-106d-40e7-8278-60520afdbb0e-c000.snappy.parquet", "Parquet/part-00000-df678a79-4a73-452b-8e72-d624b2732f17-c000.snappy.parquet"]

In [4]:
# Read the parquet files into a dataframe
df = spark.read.parquet(*parquet_files, inferSchema=True)

In [5]:
# Get unique labels and their counts
label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

# Show the results
label_counts.show()

+--------------------+-------+
|        label_tactic|  count|
+--------------------+-------+
|   Credential Access|     31|
|     Defense Evasion|      1|
|           Discovery|   2086|
|        Exfiltration|      7|
|      Initial Access|      1|
|    Lateral Movement|      4|
|         Persistence|      1|
|Privilege Escalation|     13|
|      Reconnaissance|9278722|
|Resource Development|      3|
|                none|9281599|
+--------------------+-------+



In [6]:
start_time = time.time()

# List of labels to drop
labels_to_drop = ["Defense Evasion", "Exfiltration", "Initial Access", "Lateral Movement", "Persistence", "Privilege Escalation", "Resource Development", "Credential Access", "Discovery"]

# Filter out the rows with labels to drop
df = df.filter(~col("label_tactic").isin(labels_to_drop))

# Get unique labels and their counts after filtering
filtered_label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

# Show the filtered results
filtered_label_counts.show()


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

+--------------+-------+
|  label_tactic|  count|
+--------------+-------+
|Reconnaissance|9278722|
|          none|9281599|
+--------------+-------+

Execution time: 1.7913899421691895 seconds


In [7]:
start_time = time.time()



df = df.withColumn("datetime", col("datetime").cast("string"))

# Define columns to index
columns_to_index = ['service', 'conn_state', 'history', 'proto', 'dest_ip_zeek', 'community_id', 'uid', 'src_ip_zeek', 'label_tactic', 'datetime']

# Impute null values with 'null' string
for column in columns_to_index:
    df = df.fillna('null', subset=[column])

# Apply StringIndexer to each column
indexers = [StringIndexer(inputCol=column, outputCol=column+"_indexed").fit(df) for column in columns_to_index]

# Chain indexers together
pipeline = Pipeline(stages=indexers)

# Fit and transform the data
df_indexed = pipeline.fit(df).transform(df)

# Drop original columns
df_indexed = df_indexed.drop(*columns_to_index)

# Show the schema of the DataFrame
#df_indexed.show()



end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 73.24168181419373 seconds


In [8]:
# Split the data into training and test sets
start_time = time.time()

train_data, test_data = df_indexed.randomSplit([0.7, 0.3], seed=42)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.023228168487548828 seconds


In [9]:
from pyspark.ml.feature import Imputer


start_time = time.time()

# List of numeric column names
numeric_columns = ['resp_pkts', 'orig_ip_bytes', 'missed_bytes', 'duration', 'orig_pkts',
                   'resp_ip_bytes', 'dest_port_zeek', 'orig_bytes', 'resp_bytes',
                   'src_port_zeek', 'ts']


# Create an Imputer object
imputer = Imputer(
    inputCols=numeric_columns,
    outputCols=["{}_imputed".format(column) for column in numeric_columns]
)

# Fit the imputer to the training data
imputer_model = imputer.setStrategy("mean").fit(train_data)

# Apply the imputer to the training data
train_data_imputed = imputer_model.transform(train_data)


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


start_time = time.time()


# Apply the imputer to the test data
test_data_imputed = imputer_model.transform(test_data)

# Show updated DataFrames
#train_data_imputed.show()
#test_data_imputed.show()



end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/04/15 02:33:19 WARN DAGScheduler: Broadcasting large task binary with size 268.3 MiB


Execution time: 64.01179528236389 seconds
Execution time: 0.018826723098754883 seconds


In [10]:
from pyspark.ml.feature import VectorAssembler


start_time = time.time()



# List of columns to assemble
columns_to_assemble = [column for column in train_data_imputed.columns if column.endswith("_imputed")]

# Create the VectorAssembler
assembler = VectorAssembler(inputCols=columns_to_assemble, outputCol="features")

# Transform the training DataFrame
train_data_assembled = assembler.transform(train_data_imputed)


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


start_time = time.time()
# Transform the test DataFrame
test_data_assembled = assembler.transform(test_data_imputed)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")



# Select only the features and label columns for both training and test sets
train_data_assembled = train_data_assembled.select("features", "label_tactic_indexed")
test_data_assembled = test_data_assembled.select("features", "label_tactic_indexed")

# Show the schema of the assembled training DataFrame
#train_data_assembled.printSchema()

# Show the schema of the assembled test DataFrame
#test_data_assembled.printSchema()





Execution time: 4.771420001983643 seconds
Execution time: 0.08549022674560547 seconds


In [11]:
from pyspark.ml.feature import StandardScaler


start_time = time.time()

# Standardize data on the training set
scaler = StandardScaler(inputCol="features", outputCol="features_normalized", withMean=True, withStd=True)
scaler_model = scaler.fit(train_data_assembled)
train_data_normalized = scaler_model.transform(train_data_assembled)
train_data_normalized = train_data_normalized.select("features_normalized", "label_tactic_indexed")


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/04/15 02:34:28 WARN DAGScheduler: Broadcasting large task binary with size 268.3 MiB
24/04/15 02:35:49 WARN DAGScheduler: Broadcasting large task binary with size 268.2 MiB


Execution time: 97.81581473350525 seconds


In [12]:
start_time = time.time()


# Apply the same transformation to the test set
test_data_normalized = scaler_model.transform(test_data_assembled)
test_data_normalized = test_data_normalized.select("features_normalized", "label_tactic_indexed")


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.036725521087646484 seconds


In [13]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import pandas as pd

start_time = time.time()

# Convert Spark DataFrame to Pandas DataFrame
train_pd = train_data_normalized.toPandas()
test_pd = test_data_normalized.toPandas()

# Extract features and labels from Pandas DataFrames
X_train = train_pd['features_normalized'].values.tolist()
y_train = train_pd['label_tactic_indexed'].values.tolist()
X_test = test_pd['features_normalized'].values.tolist()
y_test = test_pd['label_tactic_indexed'].values.tolist()


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")



# Define the number of components for LDA
n_components = 1 

start_time = time.time()

# Perform Linear Discriminant Analysis in scikit-learn with the specified number of components
lda = LinearDiscriminantAnalysis(n_components=n_components)
X_train_lda = lda.fit_transform(X_train, y_train)
X_test_lda = lda.transform(X_test)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


start_time = time.time()

# Convert the transformed arrays back to Pandas DataFrames
train_pd_lda = pd.DataFrame(X_train_lda, columns=[f'lda_feature_{i+1}' for i in range(n_components)])
test_pd_lda = pd.DataFrame(X_test_lda, columns=[f'lda_feature_{i+1}' for i in range(n_components)])

# Combine the transformed features with the labels
train_pd_lda['label_tactic_indexed'] = y_train
test_pd_lda['label_tactic_indexed'] = y_test

# Convert Pandas DataFrames back to Spark DataFrames
train_lda = spark.createDataFrame(train_pd_lda)
test_lda = spark.createDataFrame(test_pd_lda)


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Show the adjusted data
#train_lda.show()
#test_lda.show()

24/04/15 02:36:18 WARN DAGScheduler: Broadcasting large task binary with size 268.3 MiB
24/04/15 02:42:02 WARN DAGScheduler: Broadcasting large task binary with size 268.3 MiB


Execution time: 521.8883593082428 seconds
Execution time: 111.16402387619019 seconds
Execution time: 340.2330536842346 seconds


In [14]:
start_time = time.time()

# List of columns to assemble
columns_to_assemble = [f'lda_feature_{i+1}' for i in range(n_components)]

# Create the VectorAssembler
assembler = VectorAssembler(inputCols=columns_to_assemble, outputCol="features")

# Transform train_lda
train_lda_assembled = assembler.transform(train_lda)

# Transform test_lda
test_lda_assembled = assembler.transform(test_lda)

# Select only the assembled features and label column for both datasets
train_lda_assembled = train_lda_assembled.select("features", "label_tactic_indexed")
test_lda_assembled = test_lda_assembled.select("features", "label_tactic_indexed")

# Show the schema of the assembled train_lda DataFrame
train_lda_assembled.printSchema()

# Show the schema of the assembled test_lda DataFrame
test_lda_assembled.printSchema()


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

root
 |-- features: vector (nullable = true)
 |-- label_tactic_indexed: double (nullable = true)

root
 |-- features: vector (nullable = true)
 |-- label_tactic_indexed: double (nullable = true)

Execution time: 0.0645287036895752 seconds


In [15]:
train_lda_assembled.show()
test_lda_assembled.show()

24/04/15 02:52:07 WARN TaskSetManager: Stage 48 contains a task of very large size (31770 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 02:52:12 WARN PythonRunner: Detected deadlock while completing task 0.0 in stage 48 (TID 151): Attempting to kill Python Worker
24/04/15 02:52:12 WARN TaskSetManager: Stage 49 contains a task of very large size (13605 KiB). The maximum recommended task size is 1000 KiB.


+--------------------+--------------------+
|            features|label_tactic_indexed|
+--------------------+--------------------+
|[-2.414046284916492]|                 0.0|
|[-1.9726993004893...|                 0.0|
|[-1.9423208510779...|                 0.0|
| [-1.95764993128877]|                 0.0|
|[-2.7283160927273...|                 0.0|
|[-2.728307116964097]|                 0.0|
|[-2.7266503199313...|                 0.0|
|[-2.7264645597819...|                 0.0|
|[-2.7264645597819...|                 0.0|
|[-2.7283048166329...|                 0.0|
|[-2.7283018849788...|                 0.0|
|[-2.7283018849788...|                 0.0|
|[-2.7266463719530...|                 0.0|
|[-2.7266424236540...|                 0.0|
|[-2.7284223306159...|                 0.0|
|[-2.7284223306159...|                 0.0|
|[-2.7284122573926...|                 0.0|
|[-2.727704448802508]|                 0.0|
|[-2.728421687135104]|                 0.0|
|[-2.7284200303689...|          

+--------------------+--------------------+
|            features|label_tactic_indexed|
+--------------------+--------------------+
|[-1.924423979948703]|                 0.0|
|[-2.7283160927273...|                 0.0|
|[-2.7283043763405...|                 0.0|
|[-2.7283043763405...|                 0.0|
|[-2.7283137924454...|                 0.0|
|[-2.7283137924454...|                 0.0|
|[-2.7283048166329...|                 0.0|
|[-2.7266463719530...|                 0.0|
|[-2.7266424236540...|                 0.0|
|[-2.728423987444459]|                 0.0|
|[-2.728423987444459]|                 0.0|
|[-2.7284122573926...|                 0.0|
|[-2.728411527925069]|                 0.0|
|[-2.728411527925069]|                 0.0|
|[-2.727704448802508]|                 0.0|
|[-2.728421687135104]|                 0.0|
|[-2.7284200303689...|                 0.0|
|[-2.7284092275978...|                 0.0|
|[-2.7282864969993...|                 0.0|
|[-2.7282386093219...|          

24/04/15 02:52:16 WARN PythonRunner: Detected deadlock while completing task 0.0 in stage 49 (TID 152): Attempting to kill Python Worker


In [16]:
# Create the SVM model
start_time = time.time()
svm = LinearSVC(labelCol="label_tactic_indexed", featuresCol="features", maxIter=10, regParam=0.0, tol=.00001, fitIntercept=True)

# One Vs. Rest
ovr = OneVsRest(classifier=svm, labelCol='label_tactic_indexed')

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")



# Fit the model
start_time = time.time()

svm_model = ovr.fit(train_lda_assembled)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/04/15 02:52:16 WARN TaskSetManager: Stage 50 contains a task of very large size (31770 KiB). The maximum recommended task size is 1000 KiB.


Execution time: 0.01754927635192871 seconds


24/04/15 02:52:19 WARN TaskSetManager: Stage 53 contains a task of very large size (31770 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 02:52:24 WARN TaskSetManager: Stage 54 contains a task of very large size (31770 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 02:52:28 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/04/15 02:52:28 WARN TaskSetManager: Stage 56 contains a task of very large size (31770 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 02:52:32 WARN TaskSetManager: Stage 58 contains a task of very large size (31770 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 02:52:33 WARN TaskSetManager: Stage 60 contains a task of very large size (31770 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 02:52:33 WARN TaskSetManager: Stage 62 contains a task of very large size (31770 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 02:52:33 WARN TaskSetManag

Execution time: 34.655961751937866 seconds


In [17]:
# Make predictions
start_time = time.time()

predictions = svm_model.transform(test_lda_assembled)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.1630089282989502 seconds


In [18]:
# Evaluate the model
# Calculate accuracy
start_time = time.time()
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="accuracy")
accuracy = evaluator_accuracy.evaluate(predictions)
print("Accuracy:", accuracy)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

# Calculate precision
start_time = time.time()
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="weightedPrecision")
precision = evaluator_precision.evaluate(predictions)
print("Precision:", precision)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

# Calculate recall
start_time = time.time()
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="weightedRecall")
recall = evaluator_recall.evaluate(predictions)
print("Recall:", recall)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

# Calculate F1-score
start_time = time.time()
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="f1")
f1_score = evaluator_f1.evaluate(predictions)
print("F1-Score:", f1_score)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")



print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1_score)

24/04/15 02:52:51 WARN TaskSetManager: Stage 162 contains a task of very large size (13605 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 02:53:51 WARN TaskSetManager: Stage 164 contains a task of very large size (13605 KiB). The maximum recommended task size is 1000 KiB.


Accuracy: 1.0
Execution time: 60.4352502822876 seconds


24/04/15 02:54:50 WARN TaskSetManager: Stage 166 contains a task of very large size (13605 KiB). The maximum recommended task size is 1000 KiB.


Precision: 1.0
Execution time: 58.26351046562195 seconds


24/04/15 02:55:49 WARN TaskSetManager: Stage 168 contains a task of very large size (13605 KiB). The maximum recommended task size is 1000 KiB.


Recall: 1.0
Execution time: 59.367522954940796 seconds


F1-Score: 1.0
Execution time: 59.648566246032715 seconds
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-Score: 1.0


In [19]:
from pyspark.sql.functions import expr

start_time = time.time()

# Extract Predictions and True Labels
predictions_and_labels = predictions.select("prediction", "label_tactic_indexed")

# Calculate False Positives
false_positives = predictions_and_labels.filter((predictions_and_labels.prediction == 1) & (predictions_and_labels.label_tactic_indexed == 0)).count()

# Calculate True Negatives
true_negatives = predictions_and_labels.filter((predictions_and_labels.prediction == 0) & (predictions_and_labels.label_tactic_indexed == 0)).count()

# Calculate False Positive Rate (FPR)
fpr = false_positives / (false_positives + true_negatives)

print("False Positive Rate:", fpr)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


24/04/15 02:56:49 WARN TaskSetManager: Stage 170 contains a task of very large size (13605 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 02:57:20 WARN TaskSetManager: Stage 173 contains a task of very large size (13605 KiB). The maximum recommended task size is 1000 KiB.


False Positive Rate: 0.0
Execution time: 63.83176827430725 seconds


In [20]:
spark.sparkContext.stop()